# 🧠 Exploración del Cerebro de la Mosca (FlyEM Male CNS v1.0)

Este notebook te permite probar y analizar interactivamente el dataset del conectoma y microscopía electrónica (*Drosophila melanogaster*, Male CNS v1.0 de HHMI Janelia / Google Research).

### Contenido:
1. **Carga e inspección de anotaciones y tipos celulares**
2. **Análisis de neurotransmisores (GABA, Dopamina, Acetilcolina, etc.)**
3. **Análisis de la red de conectividad y grafos sinápticos con NetworkX**
4. **Visualización 3D de esqueletos morfológicos (SWC)**
5. **Extracción y visualización de microscopía electrónica (EM) en tiempo real con `cloud-volume`**
6. **🇺🇾 FlyTruco: La mosca jugando al Truco Uruguayo con Muestra, Piezas y Flor**

In [ ]:
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from cloudvolume import CloudVolume

# Localizar la carpeta del proyecto de forma infalible (evita errores según desde dónde se abra Jupyter)
PROJECT_ROOT = Path(r"C:\Users\nahue\OneDrive\Escritorio\UCU\Proyectos\mosca")
if not (PROJECT_ROOT / "data" / "flat-connectome").exists():
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data" / "flat-connectome"
SKELETONS_DIR = PROJECT_ROOT / "data" / "skeletons"
EM_DIR = PROJECT_ROOT / "data" / "em_cutouts"

# Permitir importar scripts locales desde cualquier ubicación
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Carpeta del proyecto: {PROJECT_ROOT}")
print(f"Directorio de datos: {DATA_DIR} (Existe: {DATA_DIR.exists()})")
print("Entorno listo. Todo funcionando correctamente.")

## 1. Cargar Anotaciones Celulares y Neurotransmisores

In [ ]:
ann = pd.read_feather(DATA_DIR / "body-annotations-male-cns-v1.0-minconf-0.5.feather")
nt = pd.read_feather(DATA_DIR / "body-neurotransmitters-male-cns-v1.0.feather")

print(f"Total neuronas anotadas: {len(ann):,}")
print(f"Total predicciones neurotransmisores: {len(nt):,}")
ann.head(5)

In [ ]:
# Distribución de neuronas por superclase
plt.figure(figsize=(10, 4))
ann['superclass'].value_counts().head(8).plot(kind='bar', color='skyblue', edgecolor='black')
plt.title("Distribución de Neuronas por Superclase")
plt.ylabel("Cantidad de neuronas")
plt.xticks(rotation=45, ha='right')
plt.show()

## 2. Grafo de Conectividad Sináptica
Cargamos la matriz de conexiones (`connectome-weights-male-cns-v1.0-minconf-0.5-traced-only.feather`).

In [ ]:
weights = pd.read_feather(DATA_DIR / "connectome-weights-male-cns-v1.0-minconf-0.5-traced-only.feather")
print(f"Total de conexiones sinápticas trazadas: {len(weights):,}")
print("Top 5 conexiones más fuertes del cerebro:")
weights.sort_values("weight", ascending=False).head(5)

In [ ]:
# Crear un subgrafo local de una neurona famosa: VS (Vertical System, 10112)
target_body = 10112
min_synapses = 15

sub_edges = weights[(
    ((weights['body_pre'] == target_body) | (weights['body_post'] == target_body))
    & (weights['weight'] >= min_synapses)
)]

G = nx.DiGraph()
for _, r in sub_edges.iterrows():
    G.add_edge(r['body_pre'], r['body_post'], weight=r['weight'])

type_map = ann.set_index('bodyId')['type'].to_dict()
labels = {n: f"{type_map.get(n, n)}\n({n})" if n == target_body else f"{type_map.get(n, n)}" for n in G.nodes()}

plt.figure(figsize=(10, 7))
pos = nx.spring_layout(G, seed=42)
colors = ['#ff5555' if n == target_body else '#77bbff' for n in G.nodes()]
nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=1400, alpha=0.9)
nx.draw_networkx_edges(G, pos, arrows=True, arrowsize=15, width=1.5, edge_color='gray')
nx.draw_networkx_labels(G, pos, labels=labels, font_size=8)
nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G, 'weight'), font_size=7)
plt.title(f"Circuito Sináptico Alrededor de la Neurona {target_body} (peso >= {min_synapses})")
plt.axis('off')
plt.show()

## 3. Visualizar Esqueleto Neuronal 3D (.swc)
Usamos `fetch_neuron_skeleton.py` para descargar y plotear en 3D la morfología de la neurona.

In [ ]:
from fetch_neuron_skeleton import download_skeleton, parse_swc, plot_skeleton_3d

# Descargar y parsear esqueleto de la neurona VS (10112)
swc_file = download_skeleton(10112, str(SKELETONS_DIR))
df_nodes = parse_swc(swc_file)
print(f"Nodos del árbol dendrítico/axonal: {len(df_nodes)}")
df_nodes.head(5)

In [ ]:
# Visualizar en 3D interactivo en el notebook
fig = plt.figure(figsize=(9, 8))
ax = fig.add_subplot(111, projection='3d')
coords = {r['id']: (r['x'], r['y'], r['z']) for _, r in df_nodes.iterrows()}

for _, r in df_nodes.iterrows():
    if r['parent'] in coords:
        p = coords[r['parent']]
        ax.plot([r['x'], p[0]], [r['y'], p[1]], [r['z'], p[2]], color='navy', alpha=0.5, lw=1)

ax.set_title("Morfología Neuronal 3D (Neurona 10112 - VS)")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
plt.show()

## 4. Microscopía Electrónica (EM) en Tiempo Real con `cloud-volume`
Descarga directa de vóxeles de microscopía a 8nm de resolución.

In [ ]:
# Conectar al volumen de microscopía en Google Storage
vol = CloudVolume('precomputed://gs://flyem-male-cns/em/em-clahe-jpeg', use_https=True)
print(f"Dimensiones totales del cerebro: {vol.shape}")
print(f"Resolución de vóxel: {vol.resolution} nm")

# Extraer un corte de 500x500 vóxeles
cutout = vol[40000:40500, 40000:40500, 20000, 0]

plt.figure(figsize=(7, 7))
plt.imshow(cutout.squeeze().T, cmap='gray')
plt.title("Microscopía Electrónica Nanométrica (Corte 500x500)")
plt.axis('off')
plt.show()

## 5. 🇺🇾 FlyTruco: La Mosca jugando al Truco Uruguayo con Reglas Oficiales
El circuito biológico real del Mushroom Body (KC -> MBON) evalúa Muestra, Piezas, Flor y Envido.

In [ ]:
from truco_engine import Card, create_deck, card_power, calculate_envido, get_effective_piezas, has_flor, calculate_flor_points
from fly_brain_agent import FlyBrainTrucoAgent

# Inicializar agente con los pesos aprendidos de FlyEM
fly = FlyBrainTrucoAgent()
fly.load_model(str(PROJECT_ROOT / 'data' / 'fly_truco_model.npz'))

# Probar mano con Flor (ej. 2 piezas: 10 de Oro y 11 de Oro, muestra 3 de Oro)
muestra = Card(3, 'Oro')
mano_flor = [Card(12, 'Copa'), Card(10, 'Oro'), Card(11, 'Oro')]

print(f">>> Muestra: {muestra}")
print(f"Mano: {mano_flor}")
print(f"¿Tiene Flor?: {has_flor(mano_flor, muestra)} (Tantos de flor: {calculate_flor_points(mano_flor, muestra)})")

# Probar mano con 1 sola pieza (Envido legítimo)
mano_envido = [Card(4, 'Oro'), Card(1, 'Espada'), Card(12, 'Copa')]
print(f"\nMano 2: {mano_envido}")
print(f"¿Tiene Flor?: {has_flor(mano_envido, muestra)}")
print(f"Envido legítimo (pieza 4 + as 1 + rey 0): {calculate_envido(mano_envido, muestra)} pts")

# Propagar en el cerebro de la mosca
st = fly.encode_state(mano_flor, muestra, [], 0, 0, False)
kc_act, mbon_act = fly.forward(st)

print(f"\nCélulas de Kenyon activadas (Sparse Firing): {(kc_act > 0).sum()} de {len(kc_act)}")
print(f"¿La mosca decide cantar Truco?: {'¡SÍ, CANTA TRUCO!' if fly.decide_truco(st) else 'No, espera'}")
carta_elegida = fly.select_card_action(mano_flor, st)
print(f"Carta que elige tirar la mosca: {mano_flor[carta_elegida]}")